# Group Activity Recognition - Baseline 1
This notebook contains the complete, self-contained implementation of Baseline 1 for Group Activity Recognition on the Volleyball dataset.

### Pipeline Components:
1. **Configurations**: Class names and indices mapping.
2. **Dataset**: Dataset class with support for mock dataset creation.
3. **Model**: ResNet-50 backbone model for global feature extraction.
4. **Utilities**: Evaluation metrics (Macro F1-score).
5. **Training/Validation**: Single epoch training/eval loops.
6. **Execution**: Training initialization and main runner function.


In [1]:
import os
import sys
import cv2
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.notebook import tqdm


## 1. Configurations
Define the categories and lookup indices for both the group activities (8 classes) and individual player actions (9 classes).


In [2]:
# Group Activity Categories (8 classes)
GROUP_ACTIVITIES = ['r_set', 'r_spike', 'r_pass', 'r_winpoint', 'l_set', 'l_spike', 'l_pass', 'l_winpoint']
# Individual Player Action Categories (9 classes)
PLAYER_ACTIONS = ['waiting', 'setting', 'spiking', 'digging', 'jumping', 'blocking', 'falling', 'moving', 'standing']

group_to_idx = {name: idx for idx, name in enumerate(GROUP_ACTIVITIES)}
action_to_idx = {name: idx for idx, name in enumerate(PLAYER_ACTIONS)}


## 2. Dataset Loader
The `VolleyBallDataset` class loads the annotations and images. If the specified dataset path does not exist, it falls back to generating synthetic mock images and annotation boxes to enable testing/debugging locally.


In [ ]:
class VolleyBallDataset(Dataset):
    def __init__(self, split='train', transform=None, data_path='./volleyball/volleyball_/videos'):
        self.data_path = data_path
        self.split = split
        self.transform = transform
        
        # Define video splits
        self.train_videos = {1, 3, 6, 7, 10, 13, 15, 16, 18, 22, 23, 31, 32, 36, 38, 39, 40, 41, 42, 48, 50, 52, 53, 54}
        self.val_videos = {0, 2, 8, 12, 17, 19, 24, 26, 27, 28, 30, 33, 46, 49, 51}
        self.test_videos = {4, 5, 9, 11, 14, 20, 21, 25, 29, 34, 35, 37, 43, 44, 45, 47}
        
        self.data = []
        self.load_data()

    def load_data(self):
        # Fallback to mock data if path doesn't exist
        if not os.path.exists(self.data_path):
            print(f"[Dataset Split: {self.split}] WARNING: Data path '{self.data_path}' not found.")
            print(f"-> Generating synthetic mock dataset for local verification/dry-run.")
            self.is_mock = True
            
            # Generate dummy video annotations
            num_mock_samples = 16 if self.split == 'train' else 8
            for idx in range(num_mock_samples):
                # Construct 12 players per frame
                parsed_players = []
                for p_idx in range(12):
                    # Distribute player boxes across court sides
                    x = 120 + p_idx * 85 if p_idx < 6 else 180 + (p_idx - 6) * 85
                    y = 200 + (p_idx % 3) * 120
                    w = 55
                    h = 110
                    # Add labels
                    action_str = PLAYER_ACTIONS[p_idx % len(PLAYER_ACTIONS)]
                    parsed_players.append({
                        'x': x,
                        'y': y,
                        'w': w,
                        'h': h,
                        'action': action_str
                    })
                
                groupLabel = GROUP_ACTIVITIES[idx % len(GROUP_ACTIVITIES)]
                annotation = {
                    'groupLabel': groupLabel,
                    'playersAnnotations': parsed_players
                }
                self.data.append((f"mock_frame_{idx}.jpg", annotation))
        else:
            self.is_mock = False
            for video_folder in os.listdir(self.data_path):
                if not video_folder.isdigit():
                    continue
                video_id = int(video_folder)
                
                # Filter by split
                if self.split == 'train' and video_id not in self.train_videos:
                    continue
                elif self.split == 'val' and video_id not in self.val_videos:
                    continue
                elif self.split == 'test' and video_id not in self.test_videos:
                    continue
                    
                video_path = os.path.join(self.data_path, video_folder)
                if not os.path.isdir(video_path):
                    continue
                annotation_file = os.path.join(video_path, 'annotations.txt')
                if not os.path.exists(annotation_file):
                    continue

                for row in open(annotation_file, 'r'):
                    frame_folder, groupLabel, *playersAnnotations = row.strip().split(' ')
                    frame_name = frame_folder.split('.')[0]
                    image_frame  = os.path.join(video_path, frame_name, frame_name + '.jpg')
                    
                    parsed_players = [{
                        'x': int(playersAnnotations[i]),
                        'y': int(playersAnnotations[i+1]),
                        'w': int(playersAnnotations[i+2]),
                        'h': int(playersAnnotations[i+3]),
                        'action': playersAnnotations[i+4]
                    } for i in range(0, len(playersAnnotations), 5)]
                    
                    annotation = {
                        'groupLabel': groupLabel,
                        'playersAnnotations': parsed_players
                    }
                    self.data.append((image_frame, annotation))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_frame, annotation = self.data[idx]
        
        # Load or generate image
        if hasattr(self, 'is_mock') and self.is_mock:
            # Create a stylized volleyball court image
            image = np.zeros((720, 1280, 3), dtype=np.uint8)
            image[:, :, 0] = 34  # Dark cyan green theme
            image[:, :, 1] = 139
            image[:, :, 2] = 34
            # Draw boundary court lines
            cv2.rectangle(image, (100, 50), (1180, 670), (255, 255, 255), 4)
            # Net line
            cv2.line(image, (640, 50), (640, 670), (200, 200, 200), 6)
            
            # Render mock players directly onto the image
            for p in annotation['playersAnnotations']:
                cv2.rectangle(image, (p['x'], p['y']), (p['x'] + p['w'], p['y'] + p['h']), (0, 0, 255), 3)
        else:
            image = cv2.imread(image_frame)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
        h_orig, w_orig = image.shape[:2]
        target_h, target_w = 720, 1280
        
        scale_x = target_w / w_orig
        scale_y = target_h / h_orig
        
        scaled_players = []
        for player in annotation['playersAnnotations']:
            x = int(player['x'] * scale_x)
            y = int(player['y'] * scale_y)
            w = int(player['w'] * scale_x)
            h = int(player['h'] * scale_y)
            
            action_str = player['action']
            # Map action label string to class index
            action_idx = action_to_idx.get(action_str.lower(), 0) if not str(action_str).isdigit() else int(action_str)
            
            scaled_players.append({
                'x': x,
                'y': y,
                'w': w,
                'h': h,
                'action': action_str,
                'action_idx': action_idx
            })
            
        group_str = annotation['groupLabel']
        # Map group label string to class index
        group_idx = group_to_idx.get(group_str.lower(), 0) if not str(group_str).isdigit() else int(group_str)
        
        scaled_annotation = {
            'groupLabel': group_str,
            'groupLabel_idx': group_idx,
            'playersAnnotations': scaled_players
        }
        
        if h_orig != target_h or w_orig != target_w:
            image = cv2.resize(image, (target_w, target_h))
            
        if self.transform:
            from PIL import Image
            image = Image.fromarray(image)
            image = self.transform(image)
            
        return image, scaled_annotation


## 3. Model Architecture
The baseline uses a pretrained `ResNet-50` backbone model. The final fully connected layer is replaced with a custom Dropout and Linear layer mapping features to the 8 group activity categories.


In [4]:
class GroupActivityRecognition(nn.Module):
    def __init__(self, num_group_classes=8, num_action_classes=9, embed_dim=512, dropout=0.3):
        super(GroupActivityRecognition, self).__init__()
        # Load ResNet-50 backbone
        try:
            self.resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        except Exception:
            self.resnet = models.resnet50(pretrained=True)
            
        # Replace the final fully connected layer of ResNet-50 with dropout and linear layer
        in_features = self.resnet.fc.in_features
        if dropout > 0:
            self.resnet.fc = nn.Sequential(
                nn.Dropout(p=dropout),
                nn.Linear(in_features, num_group_classes)
            )
        else:
            self.resnet.fc = nn.Linear(in_features, num_group_classes)
        
        # Keep num_action_classes to construct empty action predictions of the right shape
        self.num_action_classes = num_action_classes

    def forward(self, images, player_boxes=None, player_counts=None):
        """
        images: tensor of size (B, C, H, W)
        player_boxes: list of tensors (unused, kept for API compatibility)
        player_counts: list of integers (unused, kept for API compatibility)
        """
        group_output = self.resnet(images)
        # Return empty action predictions for compatibility with training/validation loops
        action_output = torch.zeros(0, self.num_action_classes, device=images.device)
        return group_output, action_output


## 4. Evaluation Utilities
Helper function to compute Macro F1-score for model evaluation.


In [5]:
try:
    from sklearn.metrics import f1_score
    def compute_macro_f1(preds, targets):
        return f1_score(targets, preds, average='macro', zero_division=0)
except ImportError:
    def compute_macro_f1(preds, targets):
        classes = np.unique(np.concatenate([preds, targets]))
        if len(classes) == 0:
            return 0.0
        f1s = []
        for c in classes:
            tp = np.sum((preds == c) & (targets == c))
            fp = np.sum((preds == c) & (targets != c))
            fn = np.sum((preds != c) & (targets == c))
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            f1s.append(f1)
        return np.mean(f1s) if len(f1s) > 0 else 0.0


## 5. Training and Validation Epochs
Functions to execute a single epoch of training or validation.


In [6]:
def train_epoch(model, dataloader, criterion_group, optimizer, device, scheduler=None):
    model.train()
    epoch_group_loss = 0.0
    
    group_correct = 0
    group_total = 0
    
    all_group_preds = []
    all_group_labels = []
    
    loop = tqdm(dataloader, desc="Training", leave=False)
    for images, annotations in loop:
        images = images.to(device)
        
        # Prepare targets
        batch_group_labels = []
        for ann in annotations:
            batch_group_labels.append(ann['groupLabel_idx'])
            
        group_labels = torch.tensor(batch_group_labels, dtype=torch.long, device=device)
        
        optimizer.zero_grad()
        group_outputs, _ = model(images)
        
        # Calculate loss
        loss = criterion_group(group_outputs, group_labels)
            
        loss.backward()
        
        # Gradient clipping to stabilize training
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        if scheduler is not None:
            scheduler.step()
        
        # Track group prediction metrics
        _, pred_group = torch.max(group_outputs, 1)
        group_correct += (pred_group == group_labels).sum().item()
        group_total += group_labels.size(0)
        
        all_group_preds.extend(pred_group.cpu().numpy())
        all_group_labels.extend(group_labels.cpu().numpy())
        
        epoch_group_loss += loss.item() * images.size(0)
        
        # Display loss inside tqdm bar
        loop.set_postfix(loss=loss.item(), group_acc=group_correct/max(group_total, 1))
        
    num_samples = len(dataloader.dataset)
    group_f1 = compute_macro_f1(np.array(all_group_preds), np.array(all_group_labels))
    return {
        'loss': epoch_group_loss / num_samples,
        'acc': (group_correct / group_total) if group_total > 0 else 0.0,
        'f1': group_f1
    }


@torch.no_grad()
def val_epoch(model, dataloader, criterion_group, device):
    model.eval()
    epoch_group_loss = 0.0
    
    group_correct = 0
    group_total = 0
    
    all_group_preds = []
    all_group_labels = []
    
    for images, annotations in dataloader:
        images = images.to(device)
        
        batch_group_labels = []
        for ann in annotations:
            batch_group_labels.append(ann['groupLabel_idx'])
            
        group_labels = torch.tensor(batch_group_labels, dtype=torch.long, device=device)
        group_outputs, _ = model(images)
        
        # Calculate loss
        loss = criterion_group(group_outputs, group_labels)
            
        # Group metrics
        _, pred_group = torch.max(group_outputs, 1)
        group_correct += (pred_group == group_labels).sum().item()
        group_total += group_labels.size(0)
        
        all_group_preds.extend(pred_group.cpu().numpy())
        all_group_labels.extend(group_labels.cpu().numpy())
        
        epoch_group_loss += loss.item() * images.size(0)
        
    num_samples = len(dataloader.dataset)
    group_f1 = compute_macro_f1(np.array(all_group_preds), np.array(all_group_labels))
    return {
        'loss': epoch_group_loss / num_samples,
        'acc': (group_correct / group_total) if group_total > 0 else 0.0,
        'f1': group_f1
    }


## 6. Training Pipeline (Main Runner)
Define settings, prepare data augmentations, load validation checkpoints if available, and run the main training and validation loops.


In [ ]:
def run_baseline_1():
    epochs = 5
    batch_size = 4
    lr = 1e-4
    data_path = './volleyball/volleyball_/videos'
    dry_run = False
    
    # Override path if we enforce dry-run or if path doesn't exist
    if not os.path.exists(data_path) or dry_run:
        print("Dataset path not found or dry_run enabled. Using local mock dataset.")
        data_path = './mock_data_videos'
        
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on device: {device}")

    train_transform = transforms.Compose([
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    def collate_fn(batch):
        images = [item[0] for item in batch]
        annotations = [item[1] for item in batch]
        images = torch.stack(images, 0)
        return images, annotations
    
    # Create train, val, and test datasets
    train_dataset = VolleyBallDataset(split='train', transform=train_transform, data_path=data_path)
    val_dataset = VolleyBallDataset(split='val', transform=val_transform, data_path=data_path)
    test_dataset = VolleyBallDataset(split='test', transform=val_transform, data_path=data_path)

    # Create dataloaders
    # Note: Using num_workers=0 to avoid multiprocessing issues in Jupyter environment
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)

    print(f"Train samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    print(f"Test samples: {len(test_dataset)}")

    # Initialize model
    model = GroupActivityRecognition(
        num_group_classes=len(GROUP_ACTIVITIES),
        num_action_classes=len(PLAYER_ACTIONS),
        embed_dim=256,
        dropout=0.3
    )
    model = model.to(device)
    
    # Differential learning rates: smaller for backbone, larger for fresh head
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if 'resnet.fc' in name:
            head_params.append(param)
        else:
            backbone_params.append(param)
            
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': lr, 'weight_decay': 1e-4},
        {'params': head_params, 'lr': lr * 10, 'weight_decay': 1e-3}
    ])
    
    criterion_group = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # Cosine Annealing learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    # Checkpoint saving paths
    os.makedirs('checkpoints', exist_ok=True)
    best_model_path = 'checkpoints/best_model.pth'
    final_model_path = 'checkpoints/final_model.pth'
    best_val_f1 = 0.0

    # Load existing checkpoint weights before training
    if os.path.exists(best_model_path):
        print(f"Loading weights from existing best model checkpoint: {best_model_path}")
        try:
            model.load_state_dict(torch.load(best_model_path, map_location=device))
            print("Evaluating loaded model on validation split...")
            initial_val_metrics = val_epoch(model, val_loader, criterion_group, device)
            best_val_f1 = initial_val_metrics['f1']
            print(f"Loaded model validation F1: {best_val_f1:.4f}. Reset best_val_f1 to match.")
        except Exception as e:
            print(f"Warning: could not load or evaluate best checkpoint: {e}")
    elif os.path.exists(final_model_path):
        print(f"Loading weights from existing final model checkpoint: {final_model_path}")
        try:
            model.load_state_dict(torch.load(final_model_path, map_location=device))
            print("Evaluating loaded model on validation split...")
            initial_val_metrics = val_epoch(model, val_loader, criterion_group, device)
            best_val_f1 = initial_val_metrics['f1']
            print(f"Loaded model validation F1: {best_val_f1:.4f}. Reset best_val_f1 to match.")
        except Exception as e:
            print(f"Warning: could not load or evaluate final checkpoint: {e}")

    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }
    
    print("\nStarting Training Loop...")
    for epoch in range(1, epochs + 1):
        print(f"\n--- Epoch {epoch}/{epochs} ---")
        
        # Display learning rates
        current_lrs = [group['lr'] for group in optimizer.param_groups]
        print(f"Learning Rates - Backbone: {current_lrs[0]:.2e} | Head: {current_lrs[1]:.2e}")
        
        train_metrics = train_epoch(model, train_loader, criterion_group, optimizer, device)
        val_metrics = val_epoch(model, val_loader, criterion_group, device)
        
        # Step the scheduler
        scheduler.step()
        
        # Append history
        history['train_loss'].append(train_metrics['loss'])
        history['train_acc'].append(train_metrics['acc'])
        history['train_f1'].append(train_metrics['f1'])
        
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['acc'])
        history['val_f1'].append(val_metrics['f1'])
        
        # Dashboard printout
        print(f"Train Loss: {train_metrics['loss']:.4f} | Train Acc: {train_metrics['acc']*100:.2f}% | Train F1: {train_metrics['f1']:.4f}")
        print(f"Val   Loss: {val_metrics['loss']:.4f} | Val   Acc: {val_metrics['acc']*100:.2f}% | Val   F1: {val_metrics['f1']:.4f}")

        # Checkpoint saving logic based on validation F1 score
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            torch.save(model.state_dict(), best_model_path)
            print(f"Saved new best model to {best_model_path} with Val F1: {best_val_f1:.4f}")

    # Save final model weights
    torch.save(model.state_dict(), final_model_path)
    print(f"\nFinal model saved to {final_model_path}")
    print("Training completed successfully!")


## 7. Run Training Pipeline
Call the runner function to start the training process.


In [8]:
run_baseline_1()

Running on device: cuda
Train samples: 2152
Validation samples: 1341
Test samples: 1337
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 174MB/s] 



Starting Training Loop...

--- Epoch 1/5 ---
Learning Rates - Backbone: 1.00e-04 | Head: 1.00e-03


Training:   0%|          | 0/538 [00:00<?, ?it/s]

Train Loss: 1.2194 | Train Acc: 66.22% | Train F1: 0.3203
Val   Loss: 0.9565 | Val   Acc: 76.96% | Val   F1: 0.5859
Saved new best model to checkpoints/best_model.pth with Val F1: 0.5859

--- Epoch 2/5 ---
Learning Rates - Backbone: 9.05e-05 | Head: 9.05e-04


...

Train Loss: 0.7086 | Train Acc: 90.99% | Train F1: 0.8773
Val   Loss: 0.8019 | Val   Acc: 86.65% | Val   F1: 0.8147
Saved new best model to checkpoints/best_model.pth with Val F1: 0.8147

--- Epoch 4/5 ---
Learning Rates - Backbone: 3.52e-05 | Head: 3.46e-04


Training:   0%|          | 0/538 [00:00<?, ?it/s]

Train Loss: 0.5886 | Train Acc: 95.54% | Train F1: 0.9373
Val   Loss: 0.7823 | Val   Acc: 87.99% | Val   F1: 0.8443
Saved new best model to checkpoints/best_model.pth with Val F1: 0.8443

--- Epoch 5/5 ---
Learning Rates - Backbone: 1.05e-05 | Head: 9.64e-05


Training:   0%|          | 0/538 [00:00<?, ?it/s]

Train Loss: 0.5295 | Train Acc: 98.33% | Train F1: 0.9766
Val   Loss: 0.7641 | Val   Acc: 89.49% | Val   F1: 0.8644
Saved new best model to checkpoints/best_model.pth with Val F1: 0.8644

Final model saved to checkpoints/final_model.pth
Training completed successfully!


In [ ]:
import os
import torch
from torch.utils.data import DataLoader

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_model_path = '/kaggle/working/checkpoints/best_model.pth'

# Define transforms and collation (same as validation)
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def collate_fn(batch):
    images = [item[0] for item in batch]
    annotations = [item[1] for item in batch]
    images = torch.stack(images, 0)
    return images, annotations

# Dataset and Dataloader setup for the test split
data_path = './volleyball/volleyball_/videos'
if not os.path.exists(data_path):
    data_path = './mock_data_videos'

test_dataset = VolleyBallDataset(split='test', transform=val_transform, data_path=data_path)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)

print(f"Test samples to evaluate: {len(test_dataset)}")

# Initialize model architecture
model = GroupActivityRecognition(
    num_group_classes=len(GROUP_ACTIVITIES),
    num_action_classes=len(PLAYER_ACTIONS),
    embed_dim=256,
    dropout=0.3
)
model = model.to(device)

# Load best model weights
if os.path.exists(best_model_path):
    print(f"Loading best checkpoint weights from: {best_model_path}")
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    
    # Run evaluation
    criterion_group = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
    test_metrics = val_epoch(model, test_loader, criterion_group, device)
    
    print("\n--- Test Evaluation Results ---")
    print(f"Test Loss: {test_metrics['loss']:.4f}")
    print(f"Test Acc:  {test_metrics['acc']*100:.2f}%")
    print(f"Test F1:   {test_metrics['f1']:.4f}")
else:
    print(f"Best model checkpoint not found at: {best_model_path}. Please train the model first.")

Test samples to evaluate: 1337
Loading best checkpoint weights from: /kaggle/working/checkpoints/best_model.pth

--- Test Evaluation Results ---
Test Loss: 0.7427
Test Acc:  89.83%
Test F1:   0.8688


In [12]:
!zip -r /kaggle/working/checkpoints.zip /kaggle/working/checkpoints

  adding: kaggle/working/checkpoints/ (stored 0%)
  adding: kaggle/working/checkpoints/final_model.pth (deflated 7%)
  adding: kaggle/working/checkpoints/best_model.pth (deflated 7%)
